In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

import os
import sys
import glob

In [2]:
import glob

jars = glob.glob("/opt/spark/jars/*.jar")

spark = (
    SparkSession.builder
    .appName("Clean_users")
    .config("spark.jars", ",".join(jars))
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 19:36:55 WARN Utils: Your hostname, 2640L, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 19:36:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/08/24 19:36:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
#read excell file
folderpath = "/home/yogavarman/Projects/FoodChain/DataSet/RawData"
filename = "users"
filepath= f"{folderpath}/{filename}.xlsx"
df = (
    spark.read
        .format("excel")
        .option("header", True)
        .option("inferSchema", False)
        .option("dataAddress", "'users'!A1")
        .load(filepath)
)

In [4]:
from pyspark.sql.window import Window
window = Window.orderBy(F.monotonically_increasing_id())


df = df.withColumn(
    "user_id",
    (F.row_number().over(window) + 202608230).cast("long")
)

In [5]:
providers = ["gmail.com", "outlook.com", "zoho.com", "hotmail.com", "yahoo.com"]
df = (df
    .withColumn(
        "email",
        F.concat(
            F.lower(
                F.regexp_replace(
                    F.trim(F.col("name")),
                    r"\s+",
                    "."
                )
            ),
            F.lit("@"),
            F.element_at(
                F.array(*[F.lit(x) for x in providers]),
                (F.rand() * len(providers)).cast("int") + 1
            )
        )
    )
)

In [6]:
df = df.withColumn(
    "dob",
    F.add_months(F.current_date(), -F.col("age") * 12)
)

In [7]:
import sys

PROJECT_PATH = "/home/yogavarman/Projects/FoodChain"

sys.path.insert(0, PROJECT_PATH)

from Functions.LogInFun import generate_password, hash_password

from pyspark.sql import functions as F
from pyspark.sql.types import StringType


# Create UDF
generate_password_udf = F.udf(
    generate_password,
    StringType()
)


# Generate random password
df = df.withColumn(
    "password",
    generate_password_udf()
)


# Generate SHA-256 password hash
df = df.withColumn(
    "password_hash",
    F.sha2(F.col("password"), 256)
)

/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [8]:
df = (
    df
    .withColumn("first_name", F.split(F.trim(F.col("name")), r"\s+").getItem(0))
    .withColumn("last_name", F.split(F.trim(F.col("name")), r"\s+").getItem(1))
)

In [9]:
df=df.withColumn("username",F.col("user_id"))


In [10]:
df.printSchema()

root
 |-- user_id: long (nullable = false)
 |-- name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital Status: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- email: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- password: string (nullable = true)
 |-- password_hash: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- username: long (nullable = false)



In [11]:
df = df.select("user_id", "username", "password_hash", "first_name", "last_name", 
               "gender", "email", "dob", "password")
df=df.withColumn("dob", F.date_format(F.col("dob"), "yyyy-MM-dd"))
df = df.withColumnRenamed("dob", "date_of_birth")

In [12]:
df.printSchema()

root
 |-- user_id: long (nullable = false)
 |-- username: long (nullable = false)
 |-- password_hash: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- password: string (nullable = true)



In [13]:


from Config.db import JDBC_URL, DB_PROPERTIES, DATABASE_URL,get_conn
df.write.jdbc(
    url=JDBC_URL,
    table="foodchain.users",
    mode="append",
    properties=DB_PROPERTIES
)

26/08/24 19:37:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/24 19:37:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/24 19:37:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/24 19:37:15 WARN ExcelHeaderChecker: Number of column in Excel header is not equal to number of fields in the schema:
 Header length: 6, schema size: 3
Excel file: file:///home/yogavarman/Projects/FoodChain/DataSet/RawData/users.xlsx
26/08/24 19:37:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/24 19:37:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a s

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 2235, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
        etype, value, tb, tb_offset=tb_offset
    )
  File "/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/IPython/core/ultratb.py", line 1272, in structured_traceback
    return FormattedTB.structured_traceback(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self, etype, evalue, etb, tb_offset, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/IPython/core/ultratb.py", line 1138, in structured_traceback
    return VerboseTB.structured_traceback(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self, etype, evalue, etb, tb_offset, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/yogavarman/venvs/global_env/lib/python